# ETL Tesouro Direto — Camadas Silver e Gold (Apache Spark SQL)

Lê a camada **Bronze** (JSON gravado no S3 pelo Kafka Connect em `raw-data/kafka/<tópico>/`),
aplica limpeza/transformação (**Silver**) e gera métricas agregadas (**Gold**),
salvando ambas em Parquet no S3. Processa IPCA e PRE-FIXADOS.

Antes de rodar: confira que o `.env_spark` (na pasta `spark/`) está com suas chaves AWS.

## 1. Sessão Spark com suporte a S3

In [ ]:
import os

# Carrega o conector hadoop-aws (e o aws-java-sdk-bundle) ANTES de iniciar a JVM do Spark.
# Estas são exatamente as versões recomendadas no PDF.
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.hadoop:hadoop-aws:3.3.4 pyspark-shell"
)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_unixtime

# >>> Edite aqui o nome do seu bucket <<<
BUCKET = "desafio-pos-eng-dados-gabriel-2026"

spark = (
    SparkSession.builder
    .appName("ETL Tesouro Direto - Silver/Gold")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.endpoint", "s3.us-east-1.amazonaws.com")
    .getOrCreate()
)

print("Spark", spark.version, "pronto.")

## 2. Pipeline Bronze → Silver → Gold

A função abaixo executa as três camadas para um tipo de título (`ipca` ou `pre`):
- **Bronze**: lê o JSON bruto do S3.
- **Silver**: remove duplicações, converte os timestamps (epoch ms → data legível),
  trata valores nulos dos PUs e grava em Parquet.
- **Gold**: agrega métricas por tipo de título usando **Spark SQL** e grava em Parquet.

In [ ]:
def processar(tipo):
    topico = "postgres-dadostesouroipca" if tipo == "ipca" else "postgres-dadostesouropre"
    bronze_path = f"s3a://{BUCKET}/raw-data/kafka/{topico}/"
    silver_path = f"s3a://{BUCKET}/processed-data/{tipo}/silver/"
    gold_path   = f"s3a://{BUCKET}/analytics/{tipo}/gold/"

    # ---------- BRONZE ----------
    print(f"\n[{tipo}] Lendo Bronze: {bronze_path}")
    df_bronze = spark.read.json(bronze_path)
    df_bronze.show(5)

    # ---------- SILVER ----------
    print(f"[{tipo}] Processando Silver...")
    df_silver = (
        df_bronze.dropDuplicates()
        .withColumn("Data_Vencimento", from_unixtime(col("Data_Vencimento") / 1000, "yyyy-MM-dd"))
        .withColumn("Data_Base",       from_unixtime(col("Data_Base") / 1000, "yyyy-MM-dd"))
        .withColumn("dt_update",        from_unixtime(col("dt_update") / 1000, "yyyy-MM-dd HH:mm:ss"))
        .fillna({"PUCompraManha": 0, "PUVendaManha": 0, "PUBaseManha": 0})
    )
    df_silver.show(5)
    df_silver.write.mode("overwrite").parquet(silver_path)
    print(f"[{tipo}] Silver salva em {silver_path}")

    # ---------- GOLD (Spark SQL) ----------
    print(f"[{tipo}] Processando Gold...")
    df_silver.createOrReplaceTempView("silver")
    df_gold = spark.sql("""
        SELECT
            Tipo,
            AVG(PUCompraManha) AS Media_PUCompraManha,
            AVG(PUVendaManha)  AS Media_PUVendaManha,
            AVG(PUBaseManha)   AS Media_PUBaseManha,
            COUNT(*)           AS Total_Registros
        FROM silver
        GROUP BY Tipo
    """)
    df_gold.show()
    df_gold.write.mode("overwrite").parquet(gold_path)
    print(f"[{tipo}] Gold salva em {gold_path}")
    return df_silver, df_gold

## 3. Executar para IPCA

In [ ]:
df_silver_ipca, df_gold_ipca = processar("ipca")

## 4. Executar para PRE-FIXADOS

In [ ]:
df_silver_pre, df_gold_pre = processar("pre")

## 5. Conferência final (camada Gold)

In [ ]:
print("GOLD IPCA:")
df_gold_ipca.show()
print("GOLD PRE-FIXADOS:")
df_gold_pre.show()